# TP1 : Préparation de données - Analyse descriptive (exercice)
## Dataset : Adult Census Income (données quasi propres)

**Contexte :** Les données sont déjà presque propres. L'objectif n'est pas de nettoyer mais d'**explorer et comprendre** le jeu de données avant toute modélisation : c'est l'étape d'**Analyse Exploratoire des Données** (EDA).

**Datamap (dictionnaire des données) :**

| Colonne | Type | Description |
|---|---|---|
| `workclass` | catégorielle | Secteur d'emploi (Private, Self-emp, Federal-gov...) |
| `race` | catégorielle | Origine ethnique déclarée |
| `sex` | catégorielle | Sexe de l'individu |
| `age` | numérique | Âge en années |
| `education_num` | numérique | Niveau d'éducation encodé (nombre d'années d'études) |
| `hours_per_week` | numérique | Heures travaillées par semaine |
| `capital_gain` | numérique | Plus-value en capital déclarée |

> **Version exercice** : les cellules marquées `# TODO` sont à compléter. Le reste (imports, chargement des données, affichages) est déjà fourni.
> Installe les dépendances une seule fois avec `pip install -r requirements.txt` depuis `cours_ml/todo/` (voir le README de ce dossier). Ce TP s'inspire de `cours_ml/05_preparation_donnees/tp_dataprep_1_propre.ipynb` (même méthode, données différentes et plus volumineuses : 3000 individus contre 344 manchots).

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', None)


---
## 1. Chargement & première inspection

Avant toute analyse, on regarde **la forme, les types et un aperçu** des données.

In [ ]:
df = pd.read_csv('adults_clean.csv')

print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()


In [ ]:
df.info()

In [ ]:
# On distingue variables numériques et catégorielles : base de toute l'analyse
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print(f"Variables numériques ({len(num_cols)})  : {num_cols}")
print(f"Variables catégorielles ({len(cat_cols)}) : {cat_cols}")


In [ ]:
# Quelques valeurs manquantes résiduelles (données quasi propres)
na = df.isnull().sum()
na_pct = (na / len(df) * 100).round(1)
na_summary = pd.DataFrame({'NA': na, '% NA': na_pct})
print(na_summary[na_summary['NA'] > 0])
print(f"\nTotal : {df.isnull().sum().sum()} valeurs manquantes ({df.isnull().sum().sum()/df.size*100:.1f}% des cellules)")


---
## 2. Statistiques descriptives univariées

### 2.1 Variables numériques

Les statistiques classiques : tendance centrale (moyenne, médiane), dispersion (écart-type, IQR), forme (asymétrie, aplatissement).

In [ ]:
df[num_cols].describe().round(2)

In [ ]:
# On enrichit avec médiane, skewness (asymétrie) et kurtosis (aplatissement)
# TODO : construire un DataFrame recapitulatif avec pour chaque colonne numerique :
# moyenne, mediane, ecart-type, coefficient de variation (std/mean*100), min, max, skewness, kurtosis
# Indice : df[num_cols].skew() et df[num_cols].kurt()
stats_num = ...
stats_num


### 2.2 Variables catégorielles

Pour les catégorielles : fréquences (effectifs et proportions), modalité dominante.

In [ ]:
for col in cat_cols:
    print(f"=== {col} ===")
    counts = df[col].value_counts()
    pct = df[col].value_counts(normalize=True) * 100
    summary = pd.DataFrame({'effectif': counts, '%': pct.round(1)})
    print(summary)
    print()


---
## 3. Visualisation des distributions

### 3.1 Histogrammes (variables numériques)

L'histogramme montre la **forme de la distribution** : symétrique, asymétrique, unimodale, multimodale…

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df[col], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(df[col].mean(),   color='red',    linestyle='--', label=f'Moyenne ({df[col].mean():.1f})')
    ax.axvline(df[col].median(), color='darkorange', linestyle='--', label=f'Médiane ({df[col].median():.1f})')
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle('Distributions des variables numériques', fontsize=13)
plt.tight_layout()
plt.show()


### 3.2 Boxplots (détection visuelle de la dispersion et des extrêmes)

Le boxplot résume : médiane (trait central), quartiles Q1-Q3 (boîte), moustaches (1.5×IQR) et points au-delà (extrêmes potentiels).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax, color='lightsteelblue')
    ax.set_title(col)
    ax.set_ylabel('')
plt.suptitle('Boxplots des variables numériques', fontsize=13)
plt.tight_layout()
plt.show()


### 3.3 Boxplots par groupe (analyse bivariée num × catégorielle)

Comparer une variable numérique **selon les modalités** d'une catégorielle révèle des différences entre groupes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, num_cols):
    sns.boxplot(data=df, x='workclass', y=col, ax=ax, hue='workclass', legend=False)
    ax.set_title(f'{col} par workclass')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle("Variables numériques selon la catégorie professionnelle", fontsize=13)
plt.tight_layout()
plt.show()


### 3.4 Comptages des variables catégorielles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, ax=ax, hue=col, order=order, legend=False)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Distribution des variables catégorielles', fontsize=13)
plt.tight_layout()
plt.show()


---
## 4. Associations entre variables

Selon les types de variables, on utilise des mesures d'association différentes :

| Type 1 | Type 2 | Mesure |
|---|---|---|
| Numérique | Numérique | Corrélation de Pearson / Spearman |
| Catégorielle | Catégorielle | V de Cramér |
| Numérique | Catégorielle | Rapport de corrélation η² |

### 4.1 Corrélation de Pearson (numérique × numérique)

In [ ]:
corr = df[num_cols].corr(method='pearson')

plt.figure(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))  # masque le triangle supérieur (redondant)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, square=True, cbar_kws={'label': 'Pearson r'})
plt.title('Matrice de corrélation (Pearson)')
plt.tight_layout()
plt.show()


In [ ]:
# Pairplot : visualise toutes les relations bivariées d'un coup, coloré par sexe
sns.pairplot(df, vars=num_cols, hue='sex', diag_kind='kde', height=2)
plt.suptitle('Relations bivariées par sexe', y=1.02)
plt.show()


### 4.2 V de Cramér (catégorielle × catégorielle)

Le **V de Cramér** mesure l'association entre deux variables catégorielles, entre 0 (indépendance) et 1 (association parfaite).
Il est basé sur le test du χ² mais normalisé pour être comparable.

$$V = \sqrt{\frac{\chi^2 / n}{\min(k-1, r-1)}}$$

In [ ]:
def cramers_v(x, y):
    """Calcule le V de Cramér entre deux séries catégorielles."""
    # TODO : implementer le V de Cramer
    # 1) confusion = pd.crosstab(x, y)  2) chi2 = chi2_contingency(confusion)[0]  3) n = confusion.sum().sum()
    # 4) phi2 = chi2 / n  5) r, k = confusion.shape  6) return sqrt(phi2 / min(k-1, r-1))
    ...

# Matrice de V de Cramér entre toutes les catégorielles
cramer_matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for c1 in cat_cols:
    for c2 in cat_cols:
        cramer_matrix.loc[c1, c2] = cramers_v(df[c1], df[c2])

plt.figure(figsize=(6, 5))
sns.heatmap(cramer_matrix.astype(float), annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0, vmax=1, linewidths=0.5, square=True, cbar_kws={'label': "V de Cramér"})
plt.title("Associations catégorielles (V de Cramér)")
plt.tight_layout()
plt.show()


In [ ]:
# Table de contingence workclass × sex pour illustrer
ct = pd.crosstab(df['workclass'], df['sex'], margins=True)
print("Table de contingence workclass × sex :")
print(ct)
print(f"\nV de Cramér workclass/sex : {cramers_v(df['workclass'], df['sex']):.3f}")


### 4.3 Rapport de corrélation (numérique × catégorielle)

Le **rapport de corrélation η²** mesure quelle proportion de la variance d'une variable numérique est expliquée par une variable catégorielle (entre 0 et 1).

In [ ]:
def correlation_ratio(categories, values):
    """Rapport de corrélation eta² entre une catégorielle et une numérique."""
    # TODO : implementer eta²
    # ss_total = variance totale de values ; ss_between = somme ponderee des ecarts des moyennes de groupe a la moyenne globale
    # eta² = ss_between / ss_total
    ...

# Matrice catégorielle × numérique
eta_matrix = pd.DataFrame(index=cat_cols, columns=num_cols, dtype=float)
for cat in cat_cols:
    for num in num_cols:
        eta_matrix.loc[cat, num] = correlation_ratio(df[cat], df[num])

plt.figure(figsize=(9, 4))
sns.heatmap(eta_matrix.astype(float), annot=True, fmt='.2f', cmap='Greens',
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'η² (eta²)'})
plt.title('Rapport de corrélation (catégorielle → numérique)')
plt.tight_layout()
plt.show()
# education_num et hours_per_week sont souvent les variables les plus liees a workclass


---
## 5. Synthèse

**Démarche d'analyse descriptive (à reproduire sur tout dataset) :**
1. **Inspecter** la structure : dimensions, types, valeurs manquantes
2. **Décrire** chaque variable seule (univarié) : stats pour le numérique, fréquences pour le catégoriel
3. **Visualiser** les distributions : histogrammes et boxplots
4. **Mesurer les associations** (bivarié) avec la bonne métrique selon les types :
   - Pearson (num × num)
   - V de Cramér (cat × cat)
   - Rapport de corrélation η² (num × cat)

**Ce qu'on a appris sur ce dataset :** l'âge, le niveau d'éducation et les heures travaillées sont les variables numériques les plus structurantes ; la catégorie professionnelle (workclass) est fortement associée au sexe dans cet échantillon, un signal à creuser avant toute modélisation.

---
## Session à rendre

Cette section est à compléter et à rendre à l'issue du TP. Réponds à chaque question dans la
cellule *Réponse* juste en dessous, à partir des résultats que **tu as toi-même obtenus** en
réalisant ce notebook (il n'y a pas de réponse générique valable pour tout le monde : les valeurs
numériques, choix d'hyperparamètres et graphiques dépendent de ton exécution).

**Q1.** D'après ton tableau de statistiques enrichi (skewness/kurtosis), quelle variable numérique est la plus asymétrique (skewed) ? Qu'est-ce que cela signifie concrètement sur sa distribution ?

*Réponse :*

_(à compléter)_

**Q2.** D'après ta matrice de V de Cramér, quelle paire de variables catégorielles est la plus fortement associée ? Est-ce surprenant ou attendu ?

*Réponse :*

_(à compléter)_

**Q3.** D'après ta matrice de rapport de corrélation η², quelle variable numérique est la plus liée à workclass ? Formule une hypothèse expliquant pourquoi.

*Réponse :*

_(à compléter)_

**Q4.** En observant les boxplots par workclass, quelle catégorie professionnelle se distingue le plus nettement des autres sur au moins une variable numérique ?

*Réponse :*

_(à compléter)_